# Weather Prediction Project: Notebook 4. Model Ranking and Final Selection

This notebook is the project decision notebook. Notebook 3 broadened the search space and documented how many model families behaved on the rainfall prediction task. The role of the present notebook is narrower and more disciplined: it decides which models are retained for the remainder of the workflow.

The selection is intentionally role-based. The notebook first chooses an interpretable baseline under one shared aligned tabular setup, then compares the strongest family representatives using their retained validation-first evidence. This is more honest than pretending that every family was benchmarked on one identical representation, because some challengers are only meaningful in their own natural setup, such as temporal sequence models.

By this point the common holdout block has already been inspected during exploration, so the one-shot holdout summaries shown here should be read as post-exploration evidence rather than as a pristine single-use final test. The strongest reliability check in this notebook comes from the rolling time-aware validation section.

## Step 1. Load the retained evidence and the aligned baseline data

The notebook combines two kinds of evidence. The first is the aligned baseline table used to compare the simpler classical candidates on a shared footing. The second is the saved winner-package evidence used to summarize the strongest family representatives after the exploratory stage.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

possible_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "models").exists() and (path / "notebooks").exists()
    ),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.ines_modeling_core import (
    TIME_COL,
    chronological_split,
    chronological_validation_split,
    fit_standard_model,
    predict_proba_for_plain_model,
    score_predictions,
    tune_threshold,
)
from src.models.ines_feature_modeling_aligned import crossvalidation_resample_threshold

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 180

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
WINNER_DIR = PROJECT_ROOT / "models" / "final_winner_package"

aligned_df = pd.read_csv(PROCESSED_DIR / "rain_model_dataset_aligned.csv", parse_dates=["date"])
selected_aligned_features = [
    line.strip()
    for line in (PROCESSED_DIR / "ines_selected_top25_features_aligned.txt").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

final_results = pd.read_csv(WINNER_DIR / "final_model_comparison.csv")
rolling_windows = pd.read_csv(WINNER_DIR / "robustness_time_windows.csv")

with open(WINNER_DIR / "final_hybrid_refinement_summary.json", "r", encoding="utf-8") as file:
    refinement_summary = json.load(file)

with open(WINNER_DIR / "dense_nn_summary.json", "r", encoding="utf-8") as file:
    dense_summary = json.load(file)

with open(WINNER_DIR / "lstm_summary.json", "r", encoding="utf-8") as file:
    lstm_summary = json.load(file)

with open(WINNER_DIR / "gru_summary.json", "r", encoding="utf-8") as file:
    gru_summary = json.load(file)

with open(WINNER_DIR / "attention_summary.json", "r", encoding="utf-8") as file:
    attention_summary = json.load(file)

with open(WINNER_DIR / "retained_logistic_baseline_summary.json", "r", encoding="utf-8") as file:
    logistic_summary = json.load(file)

baseline_result = refinement_summary["baseline_result"]
dense_best = dense_summary["best_result"]
lstm_best = lstm_summary["best_result"]
gru_best = gru_summary["best_result"]
attention_best = attention_summary["best_result"]
cnn_row = final_results.loc[final_results["model_view"] == "cnn_14_dilated"].iloc[0]


**Interpretation.** This notebook is a decision notebook, not another exploration notebook. Its evidence therefore has to be read in two layers. First, the interpretable-baseline role is decided on one shared aligned representation so that Logistic Regression, Random Forest, and XGBoost are compared like for like. Second, the stronger-model role is decided by comparing the best representatives that emerged from exploration, even when their best performance came from different valid representations.

This distinction is methodologically important. Forcing every family onto one identical feature space would make the table cleaner, but it would also erase the very reason some families were explored in the first place. The notebook stays honest about what is directly comparable and what is being retained as the best family representative.

## Step 2. State the selection criteria before showing any rankings

The project does not retain models based on one metric alone. The decision must combine predictive strength, robustness, interpretability, and complementarity. Those criteria are declared first so that the final pair is justified by principle rather than by a post-hoc story.

In [2]:
selection_criteria = pd.DataFrame(
    [
        {
            "criterion": "Predictive performance",
            "how_it_is_read": "F1-score and recall matter most because rainy days are the costly positive class and class balance is imperfect.",
        },
        {
            "criterion": "Temporal robustness",
            "how_it_is_read": "Candidates should remain credible under time-aware validation rather than depending on one favorable split.",
        },
        {
            "criterion": "Interpretability",
            "how_it_is_read": "One retained model should remain simple enough to explain clearly as a baseline benchmark.",
        },
        {
            "criterion": "Complementarity",
            "how_it_is_read": "The final pair should keep two different roles, not two near-duplicate strong models.",
        },
    ]
)

display(selection_criteria)


,criterion,how_it_is_read
0,Predictive performance,F1-score and recall matter most because rainy days are the costly positive class and class balance is imperfect.
1,Temporal robustness,Candidates should remain credible under time-aware validation rather than depending on one favorable split.
2,Interpretability,One retained model should remain simple enough to explain clearly as a baseline benchmark.
3,Complementarity,"The final pair should keep two different roles, not two near-duplicate strong models."


**Interpretation.** The criteria are deliberately broader than a single leaderboard. F1 and recall matter because the task is imbalanced and missing rainy days is costly. Robustness matters because a model that wins once but drifts across time is not trustworthy for forecasting. Interpretability matters because the workflow wants a baseline that can still be explained clearly. Complementarity matters because the retained pair should not duplicate the same role.

Under those criteria, the notebook is not trying to keep the mathematically strongest and second-strongest models. It is trying to retain one model that explains the problem transparently and one model that solves it most effectively.

## Step 3. Choose the interpretable baseline on one shared aligned setup

The aligned daily feature table is the correct place to choose the baseline role because it keeps the comparison fair among the classical tabular models. Here the candidates use the same feature representation, the same chronological split, and the same validation-first threshold rule.

In [3]:
aligned_ordered = aligned_df.sort_values("date").reset_index(drop=True)
train_df, test_df = chronological_split(aligned_ordered)
model_train_df, valid_df = chronological_validation_split(train_df)

def to_binary_target(series: pd.Series) -> pd.Series:
    return series.fillna("No").eq("Yes").astype(int)

X_model_train = model_train_df[selected_aligned_features].copy()
X_valid = valid_df[selected_aligned_features].copy()
X_train_full = train_df[selected_aligned_features].copy()
X_test = test_df[selected_aligned_features].copy()
y_model_train = to_binary_target(model_train_df["rain_tomorrow"])
y_valid = to_binary_target(valid_df["rain_tomorrow"])
y_train_full = to_binary_target(train_df["rain_tomorrow"])
y_test = to_binary_target(test_df["rain_tomorrow"])


def evaluate_aligned_candidate(model_name: str, interpretability: str, role_reading: str) -> dict:
    validation_model = fit_standard_model(model_name, X_model_train, y_model_train)
    threshold, _ = tune_threshold(
        validation_model,
        X_valid,
        y_valid,
        model_name=model_name,
    )
    final_model = fit_standard_model(model_name, X_train_full, y_train_full)
    test_prob = predict_proba_for_plain_model(model_name, final_model, X_test)
    metrics = score_predictions(y_test, test_prob, threshold=threshold)
    return {
        "candidate_model": model_name,
        "feature_space": "aligned_top25_feature_table",
        "decision_threshold": threshold,
        "holdout_roc_auc": metrics["roc_auc"],
        "holdout_f1": metrics["f1"],
        "holdout_recall": metrics["recall"],
        "interpretability": interpretability,
        "baseline_role_reading": role_reading,
    }


aligned_baseline_candidates = pd.DataFrame(
    [
        evaluate_aligned_candidate(
            "Logistic Regression",
            "High",
            "Best interpretable probability baseline and the retained benchmark model.",
        ),
        evaluate_aligned_candidate(
            "Random Forest",
            "Moderate",
            "Stronger nonlinear baseline, but no longer interpretable enough for the retained baseline role.",
        ),
        evaluate_aligned_candidate(
            "XGBoost",
            "Moderate",
            "Strong classical challenger, but methodologically closer to the stronger final-model role than to the baseline role.",
        ),
    ]
).sort_values(["holdout_f1", "holdout_recall", "holdout_roc_auc"], ascending=False)

display(aligned_baseline_candidates)


,candidate_model,feature_space,decision_threshold,holdout_roc_auc,holdout_f1,holdout_recall,interpretability,baseline_role_reading
2,XGBoost,aligned_top25_feature_table,0.5800,0.8842,0.6573,0.7153,Moderate,"Strong classical challenger, but methodologically closer to the stronger final-model role than to the baseline role."
1,Random Forest,aligned_top25_feature_table,0.3000,0.8719,0.6341,0.7137,Moderate,"Stronger nonlinear baseline, but no longer interpretable enough for the retained baseline role."
0,Logistic Regression,aligned_top25_feature_table,0.5800,0.8620,0.6297,0.6862,High,Best interpretable probability baseline and the retained benchmark model.


**Interpretation.** On the aligned table, Logistic Regression is not the numerical winner: XGBoost reaches holdout ROC-AUC 0.8842 and F1 0.6573, while Random Forest reaches 0.8719 and 0.6341. Logistic Regression is retained anyway because it gives the cleanest interpretable benchmark, with holdout ROC-AUC around 0.862, F1 around 0.63, and recall around 0.686 while preserving a direct coefficient-based probability story. The retained baseline summary used later in notebooks 5 and 6 rounds to the same practical conclusion.

That distinction matters. Random Forest is less interpretable, and XGBoost is already methodologically closer to the stronger-model role than to the benchmark role. Logistic Regression therefore becomes the retained baseline not because it tops the aligned-table leaderboard, but because it best serves the explanatory function the workflow needs.

## Step 4. Compare the strongest family representatives for the final-model role

The final-model role is chosen differently. Here the notebook compares the strongest retained representative from each serious family after the exploratory stage. This is a family-level comparison under a shared validation-first discipline, not a single-feature-space bake-off. The distinction matters because sequence models and the final hybrid CatBoost winner do not naturally live on the same representation.

In [4]:
family_representatives = pd.DataFrame(
    [
        {
            "candidate_model": "CatBoost",
            "representation": "68-feature hybrid-plus-core winner representation",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": baseline_result["selection_threshold"],
            "holdout_roc_auc": baseline_result["test_roc_auc"],
            "holdout_f1": baseline_result["test_f1"],
            "holdout_recall": baseline_result["test_recall"],
            "final_role_reading": "Strongest overall final-model candidate.",
        },
        {
            "candidate_model": "XGBoost",
            "representation": "aligned_top25_feature_table",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": float(
                aligned_baseline_candidates.loc[
                    aligned_baseline_candidates["candidate_model"] == "XGBoost",
                    "decision_threshold",
                ].iloc[0]
            ),
            "holdout_roc_auc": float(
                aligned_baseline_candidates.loc[
                    aligned_baseline_candidates["candidate_model"] == "XGBoost",
                    "holdout_roc_auc",
                ].iloc[0]
            ),
            "holdout_f1": float(
                aligned_baseline_candidates.loc[
                    aligned_baseline_candidates["candidate_model"] == "XGBoost",
                    "holdout_f1",
                ].iloc[0]
            ),
            "holdout_recall": float(
                aligned_baseline_candidates.loc[
                    aligned_baseline_candidates["candidate_model"] == "XGBoost",
                    "holdout_recall",
                ].iloc[0]
            ),
            "final_role_reading": "Strongest classical non-CatBoost challenger from the aligned table.",
        },
        {
            "candidate_model": "Calibrated XGBoost study",
            "representation": "all-feature weather table with climate context and isotonic calibration",
            "selection_basis": "chronological_split_threshold_search_isotonic_calibration",
            "decision_threshold": 0.303,
            "holdout_roc_auc": 0.8881,
            "holdout_f1": 0.6700,
            "holdout_recall": 0.7300,
            "final_role_reading": "Strong calibrated boosted-tree challenger; carries forward threshold, SHAP, and calibration lessons but does not displace CatBoost.",
        },
        {
            "candidate_model": "Dense Neural Network",
            "representation": "68-feature scaled high-dimensional hybrid representation",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": dense_best["validation_threshold"],
            "holdout_roc_auc": dense_best["test_roc_auc"],
            "holdout_f1": dense_best["test_f1"],
            "holdout_recall": dense_best["test_recall"],
            "final_role_reading": "Useful static deep representative, but not stronger than CatBoost.",
        },
        {
            "candidate_model": "Temporal LSTM",
            "representation": "rolling 68-driver sequence views",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": lstm_best["validation_threshold"],
            "holdout_roc_auc": lstm_best["test_roc_auc"],
            "holdout_f1": lstm_best["test_f1"],
            "holdout_recall": lstm_best["test_recall"],
            "final_role_reading": "Strong temporal challenger, but still not a cleaner final choice than CatBoost.",
        },
        {
            "candidate_model": "Temporal GRU",
            "representation": "rolling 68-driver sequence views",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": gru_best["validation_threshold"],
            "holdout_roc_auc": gru_best["test_roc_auc"],
            "holdout_f1": gru_best["test_f1"],
            "holdout_recall": gru_best["test_recall"],
            "final_role_reading": "Another strong temporal challenger, especially on recall, but not the most convincing final solution.",
        },
        {
            "candidate_model": "Temporal Attention",
            "representation": "rolling 68-driver sequence views",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": attention_best["validation_threshold"],
            "holdout_roc_auc": attention_best["test_roc_auc"],
            "holdout_f1": attention_best["test_f1"],
            "holdout_recall": attention_best["test_recall"],
            "final_role_reading": "Nearly tied on ranking quality, but still slightly weaker on the final decision balance.",
        },
        {
            "candidate_model": "Temporal CNN",
            "representation": "rolling 68-driver sequence views",
            "selection_basis": "validation_first_chronological_split",
            "decision_threshold": cnn_row["selection_threshold"],
            "holdout_roc_auc": cnn_row["test_roc_auc"],
            "holdout_f1": cnn_row["test_f1"],
            "holdout_recall": cnn_row["test_recall"],
            "final_role_reading": "Useful temporal side experiment, but still below the retained CatBoost winner.",
        },
    ]
).sort_values(["holdout_f1", "holdout_recall", "holdout_roc_auc"], ascending=False)

display(family_representatives)

,candidate_model,representation,selection_basis,decision_threshold,holdout_roc_auc,holdout_f1,holdout_recall,final_role_reading
0,CatBoost,68-feature hybrid-plus-core winner representation,validation_first_chronological_split,0.5800,0.9016,0.6853,0.7510,Strongest overall final-model candidate.
4,Temporal LSTM,rolling 68-driver sequence views,validation_first_chronological_split,0.6000,0.9013,0.6836,0.7630,"Strong temporal challenger, but still not a cleaner final choice than CatBoost."
5,Temporal GRU,rolling 68-driver sequence views,validation_first_chronological_split,0.5800,0.9015,0.6826,0.7761,"Another strong temporal challenger, especially on recall, but not the most convincing final solution."
7,Temporal CNN,rolling 68-driver sequence views,validation_first_chronological_split,0.6000,0.8971,0.6824,0.7415,"Useful temporal side experiment, but still below the retained CatBoost winner."
6,Temporal Attention,rolling 68-driver sequence views,validation_first_chronological_split,0.6000,0.9016,0.6796,0.7629,"Nearly tied on ranking quality, but still slightly weaker on the final decision balance."
2,Calibrated XGBoost study,all-feature weather table with climate context and isotonic calibration,chronological_split_threshold_search_isotonic_calibration,0.3030,0.8881,0.6700,0.7300,"Strong calibrated boosted-tree challenger; carries forward threshold, SHAP, and calibration lessons but does not displace CatBoost."
3,Dense Neural Network,68-feature scaled high-dimensional hybrid representation,validation_first_chronological_split,0.5200,0.8995,0.6659,0.8106,"Useful static deep representative, but not stronger than CatBoost."
1,XGBoost,aligned_top25_feature_table,validation_first_chronological_split,0.5800,0.8842,0.6573,0.7153,Strongest classical non-CatBoost challenger from the aligned table.


**Interpretation.** Among the family representatives, CatBoost provides the strongest overall balance. Its holdout ROC-AUC is 0.9016 and holdout F1 is 0.6853, ahead of the LSTM at 0.6836, the GRU at 0.6826, the best CNN at 0.6824, the attention model at 0.6796, the calibrated XGBoost study at about 0.6700, the dense network at 0.6659, and aligned XGBoost at 0.6573. The margins are not huge, which is actually important: the retained winner survived credible challengers rather than a weak field.

The calibrated XGBoost study changes the story in a useful way. It shows that threshold tuning, climate-aware feature engineering, feature-importance review, SHAP, and isotonic calibration all matter. However, it still does not displace CatBoost on the final decision balance, so its role is to strengthen the reasoning behind calibration and interpretation rather than to become the retained winner.

The ranking also explains why CatBoost is preferred. It stays at the top without giving away recall, it operates on the richest retained representation, and it does so with a simpler final decision path than the sequence models. The project is therefore not choosing CatBoost because the alternatives failed; it is choosing CatBoost because the alternatives came close and still did not displace it.

<!-- INTEGRATED_SELECTION_MERGE -->
### Integrated boosted-tree calibration evidence in the winner decision

This is the explicit place where the broader boosted-tree study enters the winner decision. It should stay visible because removing it would hide one of the strongest non-retained classical results in the project.

The key decision question is straightforward: can the richer calibrated XGBoost study displace the final CatBoost package? The answer is still no, but the comparison is close enough to matter and therefore belongs inside the ranking notebook.

| Ranked evidence | Representation | Threshold | ROC-AUC | F1 | Precision | Recall | Decision |
| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| Final CatBoost winner | 68-feature hybrid-plus-core winner representation | 0.580 | 0.9016 | 0.6853 | 0.6302 | 0.7510 | Retain as final classifier. |
| Temporal LSTM | rolling 68-driver sequence views | 0.600 | 0.9013 | 0.6836 | - | 0.7630 | Strong challenger, not a cleaner winner. |
| Temporal GRU | rolling 68-driver sequence views | 0.580 | 0.9015 | 0.6826 | - | 0.7761 | Strong challenger, not a cleaner winner. |
| Temporal CNN | rolling 68-driver sequence views | 0.600 | 0.8971 | 0.6824 | 0.6320 | 0.7415 | Best CNN side experiment, still below CatBoost. |
| Calibrated XGBoost study | all-feature weather table with climate context and isotonic calibration | 0.303 | 0.8881 | 0.6700 | 0.6200 | 0.7300 | Keep as serious ranking evidence and as calibration support, but do not retain as the final winner. |
| Dense neural network | 68-feature scaled high-dimensional hybrid representation | 0.520 | 0.8995 | 0.6659 | - | 0.8106 | Recall-heavy, weaker final balance. |
| Aligned XGBoost | aligned top-25 feature table | 0.580 | 0.8842 | 0.6573 | - | 0.7153 | Strong baseline challenger, but weaker than the richer calibrated boosted-tree study and the final winner. |

This comparison is important for the storyline. The final winner does not beat only a small baseline table. It also beats a broader boosted-tree study that already includes threshold search, climate-aware feature context, SHAP analysis, and isotonic calibration. That makes the final CatBoost recommendation much harder to dismiss as a narrow in-house choice.

<!-- INTEGRATED_DEEP_UNIFIED -->
### Integrated deep-learning evidence

The merged neural evidence comes from two linked sources and should be read together rather than as competing duplicate tables. The family-representatives table above records the strongest saved neural challenger on the unified 68-feature hybrid driver set. The broader deep-learning notebook then shows what happened when several static architectures were tested on their original scaled high-dimensional surface, including an explicit TabNet prototype check.

That distinction matters because the notebook is trying to stay honest about feature spaces. The saved dense benchmark and the sequence models all come from the same 68 canonical weather drivers carried into the winner package, while the broader static notebook preserves the earlier scaled-table experiments that helped test whether other neural styles were worth keeping in the story.

| Neural evidence source | Canonical feature surface | Threshold used | Precision | Recall | F1 | ROC-AUC | Reading |
| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| Saved dense neural network benchmark | 68-feature scaled high-dimensional hybrid representation | 0.52 | 0.5651 | 0.8106 | 0.6659 | 0.8995 | Strongest saved static deep representative on the unified hybrid driver set, but still below CatBoost on final balance. |
| Sequential dense network | broader scaled high-dimensional table | 0.42 | 0.61 | 0.74 | 0.67 | 0.8869 | Strongest static deep result from the broader merged notebook, but still on a different surface than the locked winner package. |
| Wide & Deep network | broader scaled high-dimensional table | 0.59 | 0.57 | 0.76 | 0.65 | 0.8840 | Competitive recall, but not a cleaner final tradeoff. |
| ResNet-style dense model | broader scaled high-dimensional table | 0.50 | 0.52 | 0.80 | 0.63 | 0.8784 | Useful high-recall side experiment, but too many false alarms for retention. |

TabNet stays adjacent to this table as architecture coverage rather than as a ranked challenger. The source deep-learning notebook preserved the configured prototype, but not a locked scored result package, so it remains outside the quantitative neural ranking rows.

<!-- INTEGRATED_RANKING_SCREEN -->
### Additional exploratory studies screened before formal retention

Not every merged experiment belongs in the formal final ranking table. Some studies were useful exploration, but either clearly weaker, too preliminary, or not controlled enough to justify a retained-model role.

| Screened study | Representative result | Why it does not enter the retained pair |
| --- | --- | --- |
| LazyPredict automated sweep | chronological family triage that surfaced Nearest Centroid, XGBoost, SVC, and Logistic Regression for explicit reruns | Useful first-pass triage, but its aggregate default-label metrics are not directly comparable to the later rain-class threshold-aware ranking evidence. |
| LightGBM / calibrated LightGBM | ROC-AUC about `0.8897` to `0.8904`, F1 about `0.67` | Competitive boosted-tree results, but they did not produce a cleaner case than CatBoost and did not outperform the stronger calibrated XGBoost evidence enough to change the final decision. |
| Nearest Centroid | F1 about `0.53` after feature selection and resampling | Too weak to compete with the stronger classical or boosted candidates. |
| TPOT AutoML pipeline | CV score about `0.6587` | Useful search aid, but not a retained final evidence package. |
| Hierarchical / geographic clustering studies | weak target alignment or context-only value | Helpful for understanding location structure, not strong enough as final predictive models. |
| TabNet prototype | documented architecture configuration without a locked scored package | Keeps the deep-learning search honest, but it cannot enter the retained pair without reproducible comparable evaluation evidence. |

Keeping these studies visible helps the ranking notebook stay honest. The final pair is not retained because the project ignored alternatives; it is retained because the additional alternatives were explored and still did not create a stronger final case.

<!-- FULL_SELECTION_RANKING -->
### Selection-stage ranking across all merged model studies

This table is the notebook's unified selection-stage ranking map. It is not pretending that every row is directly one-to-one comparable. The representation column keeps the notebook honest about which models shared a feature surface and which ones are best-family results from different justified surfaces.

For naming consistency with notebook 2, notebook 4 now uses one canonical feature language: CatBoost stays on the 68-feature hybrid-plus-core winner representation, the saved dense benchmark stays on the 68-feature scaled high-dimensional hybrid representation, and the temporal rows are read as rolling sequence views built from the same 68 canonical weather drivers.

| Ranking tier | Models included | Canonical feature surface | Best evidence carried forward |
| --- | --- | --- | --- |
| Retained final classifier | CatBoost raw classification winner | 68-feature hybrid-plus-core winner representation | ROC-AUC `0.9016`, F1 `0.6853`, precision `0.6302`, recall `0.7510` at threshold `0.58`. |
| Closest selection-stage challengers | Temporal LSTM, Temporal GRU, Temporal Attention, and best Temporal CNN | rolling 68-driver sequence views | F1 roughly `0.6796` to `0.6836` and ROC-AUC roughly `0.8971` to `0.9016`; strong challengers, but none cleaner than CatBoost. |
| Strong merged boosted-tree challenger tier | LightGBM, calibrated LightGBM, all-feature XGBoost, calibrated XGBoost, and aligned XGBoost | broader all-feature weather table with climate context, plus the aligned top-25 baseline table | LightGBM and calibrated LightGBM hold around ROC-AUC `0.8897` to `0.8904` with F1 about `0.67`; calibrated XGBoost stays near ROC-AUC `0.8881` and F1 `0.67`; aligned XGBoost reaches ROC-AUC `0.8842` and F1 `0.6573`. |
| Static deep challenger tier | saved dense neural benchmark, sequential dense, Wide & Deep, and ResNet | 68-feature scaled high-dimensional hybrid representation and the broader scaled high-dimensional table | Saved dense benchmark reaches ROC-AUC `0.8995` and F1 `0.6659`; the broader sequential dense notebook reaches F1 about `0.67`; the remaining static deep variants stay below that level. |
| Retained interpretable benchmark | Logistic Regression | aligned top-25 feature table | ROC-AUC `0.8620`, F1 `0.6300`, precision `0.5826`, recall `0.6858` at threshold `0.58`. |
| Supporting but non-retained screening / classical / AutoML / clustering studies | LazyPredict automated sweep, Random Forest, KNN, Decision Tree, Linear SVM, Nearest Centroid, TPOT, and clustering diagnostics | study-specific surfaces documented in notebooks 2 and 3 | Useful search evidence, but weaker than the retained pair, too preliminary for direct ranking, or not competitive enough for final retention. |

TabNet remains documented architecture coverage rather than part of this ranked challenger ladder because the repository does not retain a scored TabNet result package.

This is the clean merged reading of notebook 4. The project did not pick CatBoost out of a narrow local contest. It ranked a full selection-stage search space, kept the feature surfaces explicit, and only then retained Logistic Regression for interpretability and CatBoost for final predictive strength.

## Step 5. Eliminate weaker or redundant candidates explicitly

A decision notebook is more convincing when it also records why non-retained candidates were removed. Some are removed because they are weaker. Others are removed because they are strong but methodologically redundant relative to a better retained model.

In [5]:
elimination_view = pd.DataFrame(
    [
        {
            "candidate_model": "Random Forest",
            "why_not_retained": "Improves over the simpler baseline family but does not offer the interpretability role of Logistic Regression or the stronger final-model role of CatBoost.",
        },
        {
            "candidate_model": "XGBoost",
            "why_not_retained": "Strong classical challenger, but too close in role to CatBoost while still weaker in the final-model competition.",
        },
        {
            "candidate_model": "Calibrated XGBoost study",
            "why_not_retained": "Strong calibrated boosted-tree challenger, but still below CatBoost on the final classification decision and therefore used as calibration and interpretation evidence instead.",
        },
        {
            "candidate_model": "Dense Neural Network",
            "why_not_retained": "Interesting static deep challenger, but not strong enough to replace the retained CatBoost winner.",
        },
        {
            "candidate_model": "Temporal LSTM / GRU / Attention / CNN",
            "why_not_retained": "Temporal challengers remained credible, but none combined predictive quality, robustness, and methodological simplicity better than CatBoost.",
        },
    ]
)

display(elimination_view)

,candidate_model,why_not_retained
0,Random Forest,Improves over the simpler baseline family but does not offer the interpretability role of Logistic Regression or the stronger final-model role of CatBoost.
1,XGBoost,"Strong classical challenger, but too close in role to CatBoost while still weaker in the final-model competition."
2,Calibrated XGBoost study,"Strong calibrated boosted-tree challenger, but still below CatBoost on the final classification decision and therefore used as calibration and interpretation evidence instead."
3,Dense Neural Network,"Interesting static deep challenger, but not strong enough to replace the retained CatBoost winner."
4,Temporal LSTM / GRU / Attention / CNN,"Temporal challengers remained credible, but none combined predictive quality, robustness, and methodological simplicity better than CatBoost."


**Interpretation.** The elimination step is where the merged search space becomes disciplined. Random Forest, KNN, Decision Tree, Linear SVM, and Nearest Centroid drop because they neither win the interpretability role nor the strongest-model role. The boosted-tree challengers remain serious but are still removed: aligned XGBoost is weaker than CatBoost on the final decision balance, and the richer LightGBM, calibrated LightGBM, and calibrated XGBoost studies are retained only as evidence about thresholding, calibration, and feature importance rather than as final winners.

The same logic applies to the neural families. The saved dense benchmark plus the broader sequential dense, Wide & Deep, and ResNet notebook stay visible because they were credible experiments, but none creates a clean enough gain to replace the retained CatBoost workflow. The temporal models are also removed not because they are weak, but because they add sequence complexity without overtaking CatBoost on the balance of predictive strength, stability, and final decision simplicity that the project ultimately values.

## Step 6. Validate the retained pair with time-aware evidence only

Because the data is temporal, standard random cross-validation would be misleading. It would mix earlier and later observations, leak future structure into the fitting process, and make the models look more stable than they really are. The retained pair is therefore validated with time-aware evidence only. The baseline model is checked with expanding chronological folds on the aligned baseline table, while the stronger CatBoost model is judged using the saved rolling-window validation evidence from the locked final winner workflow.

Threshold decisions are still made on validation data rather than tuned on the test set inside this stage. Because notebook 3 already inspected the common holdout block during exploration, the rolling windows here are the cleaner reliability check and the better defense against leakage and one-split luck.

In [6]:
aligned_temporal_dataset = aligned_df.sort_values("date").reset_index(drop=True)
X_temporal = aligned_temporal_dataset[selected_aligned_features].copy()
y_temporal = aligned_temporal_dataset["rain_tomorrow"].fillna("No").eq("Yes").astype(int)

logistic_cv_summary, logistic_cv_folds = crossvalidation_resample_threshold(
    X_temporal,
    y_temporal,
    model_name="Logistic Regression",
    resampling_strategies=["none"],
    n_splits=4,
)

time_aware_pair_summary = pd.DataFrame(
    [
        {
            "retained_model": "Logistic Regression",
            "representation": "aligned_top25_feature_table",
            "folds": int(logistic_cv_summary.iloc[0]["folds"]),
            "mean_f1": float(logistic_cv_summary.iloc[0]["mean_f1"]),
            "std_f1": float(logistic_cv_summary.iloc[0]["std_f1"]),
            "mean_recall": float(logistic_cv_folds["recall"].mean()),
            "std_recall": float(logistic_cv_folds["recall"].std(ddof=1)),
            "threshold_tuning_rule": "Validation only",
            "common_holdout_status": "Viewed earlier during exploration",
        },
        {
            "retained_model": "CatBoost",
            "representation": "68-feature hybrid-plus-core winner representation",
            "folds": int(len(rolling_windows)),
            "mean_f1": float(rolling_windows["test_f1"].mean()),
            "std_f1": float(rolling_windows["test_f1"].std(ddof=1)),
            "mean_recall": float(rolling_windows["test_recall"].mean()),
            "std_recall": float(rolling_windows["test_recall"].std(ddof=1)),
            "threshold_tuning_rule": "Validation only",
            "common_holdout_status": "Viewed earlier during exploration",
        },
    ]
)

display(time_aware_pair_summary)
display(
    rolling_windows[
        [
            "split_name",
            "train_start",
            "train_end",
            "valid_start",
            "valid_end",
            "test_start",
            "test_end",
            "validation_threshold",
            "test_f1",
            "test_recall",
        ]
    ]
)

,retained_model,representation,folds,mean_f1,std_f1,mean_recall,std_recall,threshold_tuning_rule,common_holdout_status
0,Logistic Regression,aligned_top25_feature_table,4,0.6405,0.0142,0.6652,0.0404,Validation only,Viewed earlier during exploration
1,CatBoost,68-feature hybrid-plus-core winner representation,4,0.6800,0.0098,0.7020,0.0417,Validation only,Viewed earlier during exploration


,split_name,train_start,train_end,valid_start,valid_end,test_start,test_end,validation_threshold,test_f1,test_recall
0,rolling_split_1,2007-11-01,2013-05-27,2013-05-27,2014-03-19,2014-03-19,2015-01-12,0.6600,0.6725,0.6525
1,rolling_split_2,2007-11-01,2014-03-19,2014-03-19,2015-01-12,2015-01-12,2015-11-10,0.5400,0.6762,0.7372
2,rolling_split_3,2007-11-01,2015-01-12,2015-01-12,2015-11-10,2015-11-10,2016-09-01,0.6200,0.6944,0.7358
3,rolling_split_4,2007-11-01,2015-11-10,2015-11-10,2016-09-01,2016-09-01,2017-06-25,0.6400,0.6770,0.6825


**Interpretation.** The time-aware validation results are the strongest reliability check in the notebook. Logistic Regression averages F1 0.6405 across the rolling folds with sample standard deviation 0.0142, while CatBoost averages 0.6800 with sample standard deviation 0.0098. CatBoost also keeps mean recall around 0.7020 with sample standard deviation 0.0417 across these future-looking folds. That is exactly what we want from a forecasting model: better average performance and slightly more stable fold-to-fold behavior.

Just as important, the workflow never tunes thresholds on the test block inside these rolling checks. Every temporal split trains strictly on earlier observations and selects its operating point on validation periods. Because the common holdout was already viewed in notebook 3, these rolling results are the stronger defense against leakage and one convenient split.

## Step 7. State the retained pair clearly

The selection notebook should end with a direct decision, not with another open question.

In [7]:
retained_models = pd.DataFrame(
    [
        {
            "retained_model": "Logistic Regression",
            "retained_role": "Interpretable baseline",
            "why_it_is_kept": "Transparent and easy to explain, while still providing a meaningful rainfall-probability benchmark.",
        },
        {
            "retained_model": "CatBoost",
            "retained_role": "Stronger final model",
            "why_it_is_kept": "Best overall balance of predictive strength, temporal robustness, and methodological suitability.",
        },
    ]
)

display(retained_models)


,retained_model,retained_role,why_it_is_kept
0,Logistic Regression,Interpretable baseline,"Transparent and easy to explain, while still providing a meaningful rainfall-probability benchmark."
1,CatBoost,Stronger final model,"Best overall balance of predictive strength, temporal robustness, and methodological suitability."


**Final decision.** The project therefore carries forward exactly two models: Logistic Regression as the interpretable baseline and CatBoost as the stronger final model. Logistic Regression remains because it gives a transparent, coefficient-based probability benchmark for `RainTomorrow`. CatBoost remains because it delivers the best overall balance of predictive strength, recall, temporal stability, and methodological fit.

All other models remain part of the documented search space, but they stop here as retained candidates. From the next notebook onward, the question is no longer which family to keep; it is how to refine the retained CatBoost configuration without reopening broad model shopping.